# **Library package**

In [ ]:
library(tidyverse)

In [ ]:
library(purrr)

# **Import data**

In [ ]:
val_dat<-readRDS("valid.rds")

# **Data Dictionary**

### Data Dictionary: Medication and Observation Tracking

| Variable | Data Type | Description |
| :--- | :--- | :--- |
| **file_id** | Integer | Unique identifier for each record. |
| **t_rifam** | Logical | Observed correct number of pills (from AI) for Rifampicin. |
| **g_rifam** | Logical | Ground truth number of pills from prescription for Rifampicin. |
| **t_pyra** | Logical | Observed correct number of pills (from AI) for Pyrazinamide. |
| **g_pyra** | Logical | Ground truth number of pills for Pyrazinamide. |
| **t_iso** | Logical | Observed correct number of pills (from AI) for Isoniazid. |
| **g_iso** | Logical | Ground truth number of pills for Isoniazid. |
| **t_etham** | Logical | Observed correct number of pills (from AI) for Ethambutol. |
| **g_etham** | Logical | Ground truth number of pills for Ethambutol. |
| **t_med** | Logical | Correctly observed all pills (from AI). |
| **g_med** | Logical | Ground truth number of all pills from presciption. |
| **t_place** | Logical | Observed placement (from AI) attribute. |
| **g_place** | Logical | Ground truth placement from human review. |
| **t_swollow** | Logical | Observed swallowing action (from AI). |
| **g_swollow** | Logical | Ground truth swallowing action from human review. |
| **t_protude** | Logical | Observed protrusion (from AI). |
| **g_protude** | Logical | Ground truth protrusion from human review. |
| **t_seq** | Logical | Observed sequence order (from AI). |
| **g_seq** | Logical | Ground truth sequence order from human review. |
| **t_move** | Logical | Correct observed for all movements (from AI). |
| **g_move** | Logical | Ground truth of all movements from human review. |

# **Run the codes**

In [ ]:
vars <- c("rifam", "pyra", "iso", "etham", "med", "place",
          "swollow", "protude", "seq", "move")

# 2. Manual calculation function in percentages using Wilson Method
calc_diag_percent <- function(var_name, data) {

  test_val <- data[[paste0("t_", var_name)]]
  gold_val <- data[[paste0("g_", var_name)]]

  # Calculate 2x2 components
  tp <- sum(test_val == TRUE & gold_val == TRUE, na.rm = TRUE)
  fp <- sum(test_val == TRUE & gold_val == FALSE, na.rm = TRUE)
  fn <- sum(test_val == FALSE & gold_val == TRUE, na.rm = TRUE)
  tn <- sum(test_val == FALSE & gold_val == FALSE, na.rm = TRUE)

  # Helper for proportions to Percent with 95% CI (Wilson Method)
  get_ci_percent <- function(x, n) {
    if (n == 0) return("N/A")

    # prop.test performs the Wilson score interval
    # correct = FALSE provides the standard Wilson method
    res <- prop.test(x, n, conf.level = 0.95, correct = FALSE)

    # Extract values
    est <- round(res$estimate * 100, 1)
    low <- round(res$conf.int[1] * 100, 1)
    high <- round(res$conf.int[2] * 100, 1)

    return(sprintf("%.1f%% (%.1f%%, %.1f%%)", est, low, high))
  }

  # Create a row for this variable
  data.frame(
    Variable = var_name,
    Sensitivity = get_ci_percent(tp, tp + fn),
    Specificity = get_ci_percent(tn, tn + fp),
    PPV = get_ci_percent(tp, tp + fp),
    NPV = get_ci_percent(tn, tn + fn),
    Accuracy = get_ci_percent(tp + tn, tp + tn + fp + fn),
    stringsAsFactors = FALSE
  )
}

# 3. Apply to all variables
final_table_percent <- map_df(vars, ~calc_diag_percent(.x, val_dat))

In [ ]:
vars <- c("rifam", "pyra", "iso", "etham", "med", "place",
          "swollow", "protude", "seq", "move")

# 2. Manual calculation function in percentages using Wilson Method
calc_diag_percent <- function(var_name, data) {

  test_val <- data[[paste0("t_", var_name)]]
  gold_val <- data[[paste0("g_", var_name)]]

  # Calculate 2x2 components
  tp <- sum(test_val == TRUE & gold_val == TRUE, na.rm = TRUE)
  fp <- sum(test_val == TRUE & gold_val == FALSE, na.rm = TRUE)
  fn <- sum(test_val == FALSE & gold_val == TRUE, na.rm = TRUE)
  tn <- sum(test_val == FALSE & gold_val == FALSE, na.rm = TRUE)

  # Helper for proportions to Percent with 95% CI (Wilson Method)
  get_ci_percent <- function(x, n) {
    if (n == 0) return("N/A")

    # prop.test performs the Wilson score interval
    # correct = FALSE provides the standard Wilson method
    res <- prop.test(x, n, conf.level = 0.95, correct = FALSE)

    # Extract values
    est <- round(res$estimate * 100, 1)
    low <- round(res$conf.int[1] * 100, 1)
    high <- round(res$conf.int[2] * 100, 1)

    return(sprintf("%.1f%% (%.1f%%, %.1f%%)", est, low, high))
  }

  # --- New Metrics Calculations ---

  # F1-Score: 2 * TP / (2 * TP + FP + FN)
  f1_denom <- (2 * tp) + fp + fn
  f1_val <- if(f1_denom > 0) (2 * tp) / f1_denom else NA
  f1_formatted <- if(is.na(f1_val)) "N/A" else sprintf("%.1f%%", f1_val * 100)

  # Create a row for this variable
  data.frame(
    Variable = var_name,
    Sensitivity = get_ci_percent(tp, tp + fn),
    Specificity = get_ci_percent(tn, tn + fp),
    Precision = get_ci_percent(tp, tp + fp),
    Accuracy = get_ci_percent(tp + tn, tp + tn + fp + fn),
    F1_Score = f1_formatted,
    stringsAsFactors = FALSE
  )
}

final_table_percent <- map_df(vars, ~calc_diag_percent(.x, val_dat))

# **Show the results**

In [ ]:
print(final_table_percent)

   Variable          Sensitivity          Specificity            Precision
1     rifam 86.2% (81.0%, 90.2%) 75.7% (59.9%, 86.6%) 95.4% (91.5%, 97.6%)
2      pyra 92.5% (87.8%, 95.5%) 95.7% (88.0%, 98.5%) 98.3% (95.1%, 99.4%)
3       iso 91.5% (87.1%, 94.5%) 96.8% (83.8%, 99.4%) 99.5% (97.3%, 99.9%)
4     etham 88.7% (83.2%, 92.6%) 70.5% (59.6%, 79.5%) 87.2% (81.6%, 91.3%)
5       med 83.7% (77.4%, 88.6%) 89.9% (81.9%, 94.6%) 93.9% (88.8%, 96.8%)
6     place 77.8% (72.0%, 82.6%) 95.2% (77.3%, 99.2%) 99.5% (97.0%, 99.9%)
7   swollow 80.7% (74.5%, 85.8%) 94.1% (85.8%, 97.7%) 97.4% (93.6%, 99.0%)
8   protude 95.8% (79.8%, 99.3%) 99.1% (96.9%, 99.8%) 92.0% (75.0%, 97.8%)
9       seq 92.7% (88.2%, 95.6%) 96.8% (89.0%, 99.1%) 98.9% (96.1%, 99.7%)
10     move 93.8% (71.7%, 98.9%) 99.2% (97.0%, 99.8%) 88.2% (65.7%, 96.7%)
               Accuracy F1_Score
1  84.7% (79.8%, 88.6%)    90.6%
2  93.3% (89.6%, 95.8%)    95.3%
3  92.2% (88.2%, 94.9%)    95.3%
4  83.1% (78.1%, 87.2%)    88.0%
5  85.9% (

Then, calculate the overall verification performance for both TAL verification and pill count.

In [ ]:
# 1. Helper function to extract and clean numeric values from your table
get_metrics <- function(df, row_name, col_name) {
  text <- df[df$Variable == row_name, col_name]
  # Extract all numbers (including decimals) using regex
  nums <- as.numeric(unlist(regmatches(text, gregexpr("[0-9.]+", text)))) / 100
  return(nums) # Returns vector: [Estimate, Lower, Upper]
}

# 2. Extract specific test data from final_table_percent
med_se  <- get_metrics(final_table_percent, "med", "Sensitivity")
med_sp  <- get_metrics(final_table_percent, "med", "Specificity")
move_se <- get_metrics(final_table_percent, "move", "Sensitivity")
move_sp <- get_metrics(final_table_percent, "move", "Specificity")

# Calculate Prevalence based on 'med' metrics to keep accuracy consistent
med_acc <- get_metrics(final_table_percent, "med", "Accuracy")[1]
prev    <- (med_acc - med_sp[1]) / (med_se[1] - med_sp[1])

# 3. Calculate Point Estimates (Series/AND Rule)
# Sensitivity is now the product; Specificity is now the union
comb_se  <- med_se[1] * move_se[1]
comb_sp  <- med_sp[1] + move_sp[1] - (med_sp[1] * move_sp[1])
comb_acc <- (comb_se * prev) + (comb_sp * (1 - prev))

# Precision (PPV) Calculation based on Bayes Theorem
prob_pos <- (comb_se * prev) + ((1 - comb_sp) * (1 - prev))
comb_ppv <- (comb_se * prev) / prob_pos

# F1 Score Calculation (Harmonic mean of Precision and Sensitivity)
comb_f1 <- if((comb_ppv + comb_se) > 0) {
  (2 * comb_ppv * comb_se) / (comb_ppv + comb_se)
} else {
  0
}

# 4. Deterministic Error Propagation (Delta Method)
# SE = (Upper - Lower) / 3.92
se_se1 <- (med_se[3] - med_se[2]) / 3.92
se_sp1 <- (med_sp[3] - med_sp[2]) / 3.92
se_se2 <- (move_se[3] - move_se[2]) / 3.92
se_sp2 <- (move_sp[3] - move_sp[2]) / 3.92

# Variance for Sensitivity (Product Rule: Var(AB))
var_se <- (med_se[1]^2 * se_se2^2) + (move_se[1]^2 * se_se1^2)
se_comb_se <- sqrt(var_se)

# Variance for Specificity (AND Rule complement union: Var(1 - (1-A)(1-B)))
var_sp <- ((1 - med_sp[1])^2 * se_sp2^2) + ((1 - move_sp[1])^2 * se_sp1^2)
se_comb_sp <- sqrt(var_sp)

# Variance for Accuracy (Weighted Sum)
var_acc <- (prev^2 * var_se) + ((1 - prev)^2 * var_sp)
se_comb_acc <- sqrt(var_acc)

# Variance for Precision (Delta Method with Partial Derivatives)
dPPV_dSe <- (prev * (1 - comb_sp) * (1 - prev)) / (prob_pos^2)
dPPV_dSp <- (comb_se * prev * (1 - prev)) / (prob_pos^2)
var_ppv <- (dPPV_dSe^2 * var_se) + (dPPV_dSp^2 * var_sp)
se_comb_ppv <- sqrt(var_ppv)

# 5. Result Formatting
series_results <- data.frame(
  Metric   = c("Sensitivity", "Specificity", "Accuracy", "Precision", "F1_Score"),
  Estimate = c(comb_se, comb_sp, comb_acc, comb_ppv, comb_f1),
  Lower    = c(comb_se - 1.96 * se_comb_se, comb_sp - 1.96 * se_comb_sp, comb_acc - 1.96 * se_comb_acc, comb_ppv - 1.96 * se_comb_ppv, NA),
  Upper    = c(comb_se + 1.96 * se_comb_se, comb_sp + 1.96 * se_comb_sp, comb_acc + 1.96 * se_comb_acc, comb_ppv + 1.96 * se_comb_ppv, NA)
)

# Convert to percentage
series_results[,2:4] <- lapply(series_results[,2:4], function(x) x * 100)

# CONSTRAINT RULE
# Manually adjust Specificity Lower Bound to match the best individual test (move)
# This prevents the Delta Method from incorrectly showing 100% minus 100%
series_results[series_results$Metric == "Specificity", "Lower"] <- move_sp[2] * 100

# Final cleanup: Cap all values at [0, 100], while safely ignoring the NAs in the F1 row
series_results[,2:4] <- lapply(series_results[,2:4], function(x) {
  ifelse(is.na(x), NA, pmin(pmax(x, 0), 100))
})

print(series_results)

       Metric Estimate    Lower     Upper
1 Sensitivity 78.51060 65.97389  91.04731
2 Specificity 99.91920 97.00000 100.00000
3    Accuracy 86.10720 78.01882  94.19558
4   Precision 99.94343 99.83790 100.00000
5    F1_Score 87.93994       NA        NA


In [ ]:
table(val_dat$t_rifam,val_dat$g_rifam*1)

       
          0   1
  FALSE  28  30
  TRUE    9 188

In [ ]:
table(val_dat$t_pyra,val_dat$g_pyra*1)

       
          0   1
  FALSE  66  14
  TRUE    3 172

In [ ]:
table(val_dat$t_iso,val_dat$g_iso*1)

       
          0   1
  FALSE  30  19
  TRUE    1 205

In [ ]:
table(val_dat$t_etham,val_dat$g_etham*1)

       
          0   1
  FALSE  55  20
  TRUE   23 157

In [ ]:
table(val_dat$t_med,val_dat$g_med*1)

       
          0   1
  FALSE  80  27
  TRUE    9 139

In [ ]:
table(val_dat$t_place,val_dat$g_place*1)

       
          0   1
  FALSE  20  52
  TRUE    1 182

In [ ]:
table(val_dat$t_swollow,val_dat$g_swollow*1)

       
          0   1
  FALSE  64  36
  TRUE    4 151

In [ ]:
table(val_dat$t_protude,val_dat$g_protude*1)

       
          0   1
  FALSE 229   1
  TRUE    2  23

In [ ]:
table(val_dat$t_seq,val_dat$g_seq*1)

       
          0   1
  FALSE  60  14
  TRUE    2 179

In [ ]:
table(val_dat$t_move,val_dat$g_move*1)

       
          0   1
  FALSE 237   1
  TRUE    2  15

In [ ]:
table(val_dat$g_swollow)


FALSE  TRUE 
   68   187 

In [ ]:
table(val_dat$g_protude)


FALSE  TRUE 
  231    24 

In [ ]:
table(val_dat$g_seq)


FALSE  TRUE 
   62   193 

In [ ]:
table(val_dat$g_move)


FALSE  TRUE 
  239    16 